# 13 — Retrieval-Augmented Question Answering (RAG)

## Goal
Retrieve evidence first, then use Gemini to explain it in the farmer's language.

**What happens using this code?** It searches the approved document directory, builds a prompt containing the best evidence chunks, and calls Gemini only when the API key is configured. If there are no documents, it refuses to generate a definitive farming recommendation.

In [ ]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from tools.search_agricultural_knowledge import search_agricultural_knowledge

question = "වී පැළවල කොළ කහ වෙනවා. මම කළ යුත්තේ කුමක්ද?"
evidence = search_agricultural_knowledge(question, k=4)
if not evidence:
    answer = "මෙම ප්‍රශ්නයට ගැළපෙන සත්‍යාපිත ලේඛන හමු නොවීය. නිශ්චිත ප්‍රතිකාරයක් නිර්දේශ කළ නොහැක. ප්‍රදේශයේ කෘෂිකර්ම නිලධාරියෙකුගෙන් උපදෙස් ලබාගන්න."
    print(answer)
elif not os.getenv("GEMINI_API_KEY"):
    print("Evidence found but Gemini is not configured. Review these sources:")
    for item in evidence: print(item["source"], item["score"], item["text"][:350])
else:
    from google import genai
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
    model_name = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
    context = "\n\n".join(f"Source: {x['source']}\nText: {x['text']}" for x in evidence)
    prompt = f"""Answer in the language of the question. Be cautious. Only use the sources below for agricultural claims. If the sources do not support a conclusion, say that you cannot determine the cause and refer the farmer to an agricultural officer. Never invent pesticide doses or fertilizer rates. Cite source filenames.
Question: {question}
Sources:\n{context}"""
    response = client.models.generate_content(model=model_name, contents=prompt)
    print(response.text)
    print("\nRetrieved evidence:", [(x["source"], round(x["score"], 3)) for x in evidence])

**Safety rule:** do not load the supplied synthetic Q&A file as the authoritative RAG knowledge base. Put official documents in `knowledge_base/documents/` after verifying source, date, and content with an agricultural expert.